In [1]:
from ase.optimize import FIRE  
from ase import Atoms
from ase.optimize import BFGS
from ase import Atoms
from ase.geometry import cellpar_to_cell

from pymatgen.core import Lattice, Structure
from pymatgen.core import Structure
from pathlib import Path
from pymatgen.io.ase import AseAtomsAdaptor
import numpy as np



In [2]:
from tensorpotential.calculator.foundation_models import grace_fm, GRACEModels


[tensorpotential] Info: Environment variable TF_USE_LEGACY_KERAS is automatically set to '1'.


2025-12-04 15:00:46.204058: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764849646.222035  256100 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764849646.227744  256100 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764849646.243970  256100 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764849646.243987  256100 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764849646.243989  256100 computation_placer.cc:177] computation placer alr

In [3]:
data_dir = {
    "layered_p1": {"path": "/home/a.burov/icys_2025/niohf/initial_no_relaxation/layered/layered_P1.POSCAR"},
    "layered_p-1": {"path": "/home/a.burov/icys_2025/niohf/initial_no_relaxation/layered/layered_P-1.POSCAR"},
    "layered_p21": {"path": "/home/a.burov/icys_2025/niohf/initial_no_relaxation/layered/layered_P2_1.POSCAR"},
    "layered_p2c": {"path": "/home/a.burov/icys_2025/niohf/initial_no_relaxation/layered/layered_Pc.POSCAR"},
    "layered_p21c": {"path": "/home/a.burov/icys_2025/niohf/initial_no_relaxation/layered/layered_P2_1_c.POSCAR"},
    
    "alpha": {"path": "/home/a.burov/icys_2025/niohf/initial_no_relaxation/diaspore_NiOHF_alpha.POSCAR"}, 
    "beta": {"path": "/home/a.burov/icys_2025/niohf/initial_no_relaxation/diaspore_NiOHF_beta.POSCAR"}, 
    "gamma": {"path": "/home/a.burov/icys_2025/niohf/initial_no_relaxation/diaspore_NiOHF_gamma.POSCAR"}, 
    "delta": {"path": "/home/a.burov/icys_2025/niohf/initial_no_relaxation/diaspore_NiOHF_delta.POSCAR"},      
}


In [4]:
def grace_calc(st, st_name, path_save, max_force=0.05, steps=20):
    """
    INPUT:
        st (ase Structure)
    RETURN:
        optimized structure with same atom ordering
    """
    
    # Store original positions and species
    original_symbols = st.get_chemical_symbols()
    original_positions = st.get_scaled_positions().copy()
    
    # Assign calculator
    st.calc = grace_model
    
    # Optimize
    optim = FIRE(st, dt=0.05, maxstep=0.5)
    optim.run(fmax=max_force, steps=steps)

    # Check if order changed
    new_symbols = st.get_chemical_symbols()
    if new_symbols != original_symbols:
        print(f"WARNING: Atom order changed for {st_name}!")
        
        # Try to restore order by matching positions
        from scipy.optimize import linear_sum_assignment
        
        new_positions = st.get_scaled_positions()
        n_atoms = len(st)
        dist_matrix = np.zeros((n_atoms, n_atoms))
        
        for i in range(n_atoms):
            for j in range(n_atoms):
                diff = original_positions[i] - new_positions[j]
                diff = diff - np.round(diff)
                dist_matrix[i, j] = np.linalg.norm(diff)
        
        _, col_ind = linear_sum_assignment(dist_matrix)
        
        # Reorder
        st = st[col_ind]
        print(f"  Reordered atoms for {st_name}")
    
    # Save
    st.write(path_save + f"/{st_name}.cif")
    st.write(path_save + f"/{st_name}.xyz")    
    st.write(path_save + f"/{st_name}.vasp")    
    
    return st, optim


In [5]:
# path to save data
path_save = "/home/a.burov/icys_2025/niohf/optimized/"

In [6]:
# create directory for files if it does not exist
Path(f"{path_save}").mkdir(parents=True, exist_ok=True)

# create directories for each type of calculations if they do not exist
Path(f"{path_save}/grace").mkdir(parents=True, exist_ok=True)


In [7]:
path_save_grace = path_save + "/grace/"


In [8]:
grace_model = grace_fm("GRACE-2L-OMAT-large-ft-AM",
                pad_atoms_number=10,
                pad_neighbors_fraction=0.05, 
                min_dist=0.5
               )


Using cached GRACE model from /home/a.burov/.cache/grace/GRACE-2L-OMAT-large-ft-AM
Model license: Academic Software License


2025-12-04 15:00:53.191525: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [9]:
for name, vals in data_dir.items():
    path_file = vals['path']

    st_pmg = Structure.from_file(path_file)
    st_pmg.replace_species({"Cu": "Ni"})
    
    # Store original ordering information
    st_pmg.add_site_property('original_index', list(range(len(st_pmg))))

    # Convert to ASE
    st_ase = AseAtomsAdaptor.get_atoms(st_pmg)
    
    # Store original indices as tags
    st_ase.set_tags(np.arange(len(st_ase)))

    st_ase.write(path_save + f"/dft/{name}.xyz")    
    st_ase.write(path_save + f"dft/{name}.cif")    
    st_ase.write(path_save + f"dft/{name}.vasp")    
    
    # Run calculations
    st_relaxed_grace, optim = grace_calc(st_ase.copy(), name, path_save + "/grace", steps=300)
    
    data_dir[name]['energy_grace'] = st_relaxed_grace.get_potential_energy()
    


/tmp/ipykernel_256100/2347650314.py:5: UserWarning: Some species to be substituted are not present in structure. Pls check your input. Species to be substituted = {Element Cu}; Species in structure = {Element O, Element H, Element Ni, Element F}
  st_pmg.replace_species({"Cu": "Ni"})
I0000 00:00:1764849659.328206  256100 service.cc:152] XLA service 0x5599fa074e30 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1764849659.328260  256100 service.cc:160]   StreamExecutor device (0): Host, Default Version
2025-12-04 15:00:59.617347: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1764849661.558720  256100 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


      Step     Time          Energy          fmax
FIRE:    0 15:01:01      -61.874401       27.644117
      Step     Time          Energy          fmax
FIRE:    0 15:01:02      -58.152373       27.738849
      Step     Time          Energy          fmax
FIRE:    0 15:01:02      -58.524007       27.536839
      Step     Time          Energy          fmax
FIRE:    0 15:01:02      -58.363430       27.717364
      Step     Time          Energy          fmax
FIRE:    0 15:01:03      -51.559232       27.569941
      Step     Time          Energy          fmax
FIRE:    0 15:01:05      -74.461567        6.436336
      Step     Time          Energy          fmax
FIRE:    0 15:01:06      -74.515432        6.862510
      Step     Time          Energy          fmax
FIRE:    0 15:01:06      -72.109860        3.101266
      Step     Time          Energy          fmax
FIRE:    0 15:01:06      -75.400576        1.394217


In [17]:
import csv


In [18]:
path_energies = '/home/a.burov/icys_2025/niohf/optimized/energies_grace.csv'


In [19]:
# choose keys and order
fieldnames = ['name', 'energy_grace']


rows = []

for key, val in data_dir.items():
    rows.append({
        'key': key,
        'energy_grace': val['energy_grace'],
    })


with open(path_energies, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(fieldnames)
    for key, val in data_dir.items():
        w.writerow([key, val['energy_grace'], ])
        

    

In [20]:
print(f"CSV written to: {path_energies}")


CSV written to: /home/a.burov/icys_2025/niohf/optimized/energies_grace.csv
